# Intrinsics: Add M1M3 Thermocouple Data (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-23  
**Last Modified:** 2026-04-28  
**Status:** In Progress  
**Keywords:** intrinsics, M1M3, thermocouples, EFD, visit_info  

## Description

Augment existing `{stem}_visits.parquet` files with per-thermocouple
temperatures (`m1m3_tc_<name>`) and per-cell vertical gradients
(`m1m3_dt_<name>`) extracted from `lsst.ts.m1m3.utils.ThermocoupleAnalysis`,
without re-running the full mktable pipeline.

Key functionality:
1. Discover existing `{stem}_visits.parquet` files in the AOS output directory whose date range overlaps `[DAY_OBS_MIN, DAY_OBS_MAX]`.
2. For each, query ConsDB for `obs_start`/`obs_end` and run `get_m1m3_data` per `day_obs`.
3. Merge the new `m1m3_tc_*` and `m1m3_dt_*` columns into the visits parquet (in place or sidecar).
4. Write a single global `output/m1m3_thermocouples.parquet` lookup table with thermocouple positions (one file shared across all chunks).

**Output:**
- Augmented `{stem}_visits.parquet` files (or `{stem}_visits_m1m3.parquet` sidecars)
- Global `output/m1m3_thermocouples.parquet` (name, x, y, z, core_location, scanner)

**Based on:** `intrinsics_addthermal.ipynb` pattern; consumes `intrinsics_lib.get_m1m3_data` added on 2026-04-23.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-23 | Aaron Roodman | Initial version: adds per-thermocouple temperatures and per-cell vertical gradients to existing visits parquets without re-running mktable. |
| 2026-04-28 | Aaron Roodman | Switched static thermocouple metadata to a single global `output/m1m3_thermocouples.parquet` (was per-chunk, identical content). MERGE_IN_PLACE default now False. EFD per-thermocouple values cast to float before interpolation. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Date range over which to augment existing visits parquets. Files whose
# {day_obs_min}_{day_obs_max} suffix overlaps this range are processed.
DAY_OBS_MIN = 20260315
DAY_OBS_MAX = 20260423

# Directory holding {stem}_visits.parquet files (relative to aos/).
OUTPUT_DIR = 'output'

# If True, augmented columns are written back into each {stem}_visits.parquet.
# If False (default), they go into a separate {stem}_visits_m1m3.parquet sidecar
# so the original visits parquet stays untouched.
MERGE_IN_PLACE = False

# Write the global static thermocouple lookup table at
# output/m1m3_thermocouples.parquet (one file shared by all chunks).
WRITE_METADATA_SIDECAR = True

# ConsDB endpoint. The internal Kubernetes URL works on the RSP; from outside,
# use https://usdf-rsp.slac.stanford.edu/consdb (token from ~/.lsst/consdb_token).
CONSDB_URL = 'http://consdb-pq.consdb:8080/consdb'

# Set True to recompute m1m3_* columns even when they already exist.
OVERWRITE = False

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
import glob
from pathlib import Path
from collections import OrderedDict
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Add aos/code/ to path and import the M1M3 helpers
sys.path.insert(0, 'code')
from intrinsics_lib import get_m1m3_data, _thermocouple_metadata_table
from lsst.summit.utils import ConsDbClient
from lsst.summit.utils.efdUtils import makeEfdClient

# Setup ConsDB no_proxy + token (mirrors run_mktable behavior)
os.environ.setdefault('no_proxy', '')
if '.consdb' not in os.environ['no_proxy']:
    os.environ['no_proxy'] += ',.consdb'

consdb_url = CONSDB_URL
if '@' not in consdb_url and 'consdb-pq.consdb' not in consdb_url:
    token_file = Path.home() / '.lsst' / 'consdb_token'
    if token_file.exists():
        token = token_file.read_text().strip()
        consdb_url = consdb_url.replace('://', f'://user:{token}@', 1)
consdb_client = ConsDbClient(consdb_url)
efd_client = makeEfdClient()
print('ConsDB + EFD clients ready')

<a id='functions'></a>
## Helper Functions

In [ ]:
_DATE_RANGE_RE = re.compile(r'(\d{8})_(\d{8})_visits$')

def visits_file_range(path):
    """Parse the trailing _YYYYMMDD_YYYYMMDD_visits stem of a visits parquet.
    Returns (day_obs_min, day_obs_max) or None if the filename doesn't match.
    """
    m = _DATE_RANGE_RE.search(Path(path).stem)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


def discover_visits_files(output_dir, day_obs_min, day_obs_max):
    """Find {stem}_visits.parquet files whose date range overlaps the
    requested [day_obs_min, day_obs_max] window. Excludes _fits and
    _thermocouples sidecars.
    """
    matches = []
    for p in sorted(glob.glob(f'{output_dir}/*_visits.parquet')):
        rng = visits_file_range(p)
        if rng is None:
            continue
        f_min, f_max = rng
        if f_max < day_obs_min or f_min > day_obs_max:
            continue
        matches.append(p)
    return matches

<a id='data'></a>
## Data Access

List the visits parquets that fall within the requested date range.

In [ ]:
input_files = discover_visits_files(OUTPUT_DIR, DAY_OBS_MIN, DAY_OBS_MAX)
print(f'Date window: {DAY_OBS_MIN} – {DAY_OBS_MAX}')
print(f'Found {len(input_files)} visits parquet file(s) in {OUTPUT_DIR}/:')
for p in input_files:
    rng = visits_file_range(p)
    print(f'  {p}    [{rng[0]}–{rng[1]}]')

<a id='analysis'></a>
## Analysis

For each visits parquet:
1. Load existing `(day_obs, seq_num)` pairs.
2. Query ConsDB for `obs_start` / `obs_end`.
3. Run `get_m1m3_data` per `day_obs` (matches `run_mktable` chunking, avoids EFD timeouts).
4. Merge new columns on `(day_obs, seq_num)`; write back (or sidecar).

Static thermocouple metadata is written once to a global
`output/m1m3_thermocouples.parquet` (same content for all chunks).

In [ ]:
async def add_m1m3_to_visits(visits_path, merge_in_place=True,
                              write_metadata_sidecar=True,
                              overwrite=False):
    visits_path = Path(visits_path)
    stem = visits_path.stem.replace('_visits', '')
    out_dir = visits_path.parent
    sidecar_path = out_dir / f'{stem}_visits_m1m3.parquet'
    target = visits_path if merge_in_place else sidecar_path

    print(f'\n{"="*60}')
    print(f'Processing: {visits_path}')
    print(f'{"="*60}')

    visits = pd.read_parquet(visits_path)
    print(f'  {len(visits)} visits, {len(visits.columns)} existing columns')

    has_tc = any(c.startswith('m1m3_tc_') for c in visits.columns)
    if has_tc and not overwrite:
        print('  m1m3_tc_* columns already present — skipping (set OVERWRITE=True to redo)')
        return None

    day_obs_list = sorted(set(int(d) for d in visits['day_obs']))
    day_obs_str = ', '.join(str(d) for d in day_obs_list)
    q = f"""
        SELECT e.day_obs, e.seq_num, e.obs_start, e.obs_end
        FROM cdb_lsstcam.exposure e
        WHERE e.day_obs IN ({day_obs_str})
        ORDER BY e.day_obs, e.seq_num
    """
    consdb_df = consdb_client.query(q).to_pandas()
    print(f'  ConsDB returned {len(consdb_df)} exposure records')

    visits_keys = visits[['day_obs', 'seq_num']].astype(int)
    visit_table = visits_keys.merge(
        consdb_df[['day_obs', 'seq_num', 'obs_start', 'obs_end']],
        on=['day_obs', 'seq_num'], how='left')
    n_match = visit_table['obs_start'].notna().sum()
    print(f'  Matched obs times for {n_match}/{len(visit_table)} visits')

    valid = visit_table.dropna(subset=['obs_start', 'obs_end']).copy()
    parts = []
    for day_obs_val in sorted(valid['day_obs'].unique()):
        day_visits = valid[valid['day_obs'] == day_obs_val].reset_index(drop=True)
        print(f'  M1M3 day_obs={day_obs_val}: {len(day_visits)} visits')
        try:
            gdf = await get_m1m3_data(efd_client, day_visits,
                                      include_thermocouples=True,
                                      include_cell_gradients=True)
            gdf['day_obs'] = day_visits['day_obs'].values
            gdf['seq_num'] = day_visits['seq_num'].values
            parts.append(gdf)
        except Exception as e:
            print(f'    WARNING: failed for {day_obs_val}: {e}')

    if not parts:
        print('  No M1M3 data could be extracted — aborting this file')
        return None

    m1m3_df = pd.concat(parts, ignore_index=True, sort=False)
    new_cols = [c for c in m1m3_df.columns if c not in ('day_obs', 'seq_num')]
    n_tc = sum(1 for c in new_cols if c.startswith('m1m3_tc_'))
    n_dt = sum(1 for c in new_cols if c.startswith('m1m3_dt_'))
    n_grad = sum(1 for c in new_cols if c.endswith('_gradient'))
    print(f'  Extracted {len(new_cols)} columns: '
          f'{n_grad} gradients, {n_tc} per-thermocouple, {n_dt} per-cell')

    if merge_in_place:
        drop = [c for c in visits.columns
                if c.startswith(('m1m3_tc_', 'm1m3_dt_'))
                or c in ('x_gradient', 'y_gradient', 'z_gradient', 'radial_gradient')]
        if drop:
            visits = visits.drop(columns=drop)
        merged = visits.merge(m1m3_df, on=['day_obs', 'seq_num'], how='left')
        merged.to_parquet(target, index=False)
        print(f'  Wrote: {target} ({len(merged)} rows, {len(merged.columns)} cols)')
    else:
        m1m3_df.to_parquet(target, index=False)
        print(f'  Wrote sidecar: {target}')

    return m1m3_df


def write_thermocouple_metadata(output_dir):
    """Write the static thermocouple lookup table once, globally."""
    tc_meta = _thermocouple_metadata_table()
    if len(tc_meta) == 0:
        print('Thermocouple metadata not available (lsst.ts.m1m3.utils missing)')
        return
    meta_path = Path(output_dir) / 'm1m3_thermocouples.parquet'
    tc_meta.to_parquet(meta_path, index=False)
    print(f'Wrote global thermocouple metadata: {meta_path} '
          f'({len(tc_meta)} thermocouples)')


# Write the global thermocouple metadata once (not per-chunk — same content)
if WRITE_METADATA_SIDECAR:
    write_thermocouple_metadata(OUTPUT_DIR)

results = OrderedDict()
for path in input_files:
    results[path] = await add_m1m3_to_visits(
        path,
        merge_in_place=MERGE_IN_PLACE,
        write_metadata_sidecar=False,  # handled once globally above
        overwrite=OVERWRITE)

n_done = sum(1 for r in results.values() if r is not None)
print(f'\nDone. Processed {n_done}/{len(results)} files')

<a id='results'></a>
## Results & Plots

Quick sanity check on one of the augmented files: column counts and a sample of per-thermocouple temperatures.

In [ ]:
import matplotlib.pyplot as plt

for path in input_files[:1]:
    if MERGE_IN_PLACE:
        df = pd.read_parquet(path)
    else:
        sidecar = path.replace('_visits.parquet', '_visits_m1m3.parquet')
        if not Path(sidecar).exists():
            continue
        df = pd.read_parquet(sidecar)

    tc_cols = sorted([c for c in df.columns if c.startswith('m1m3_tc_')])
    dt_cols = sorted([c for c in df.columns if c.startswith('m1m3_dt_')])
    print(f'{path}')
    print(f'  Total columns: {len(df.columns)}')
    print(f'  m1m3_tc_*: {len(tc_cols)}')
    print(f'  m1m3_dt_*: {len(dt_cols)}')

    if tc_cols:
        n_valid = df[tc_cols].notna().sum(axis=1)
        print(f'  Valid m1m3_tc count per visit: min={n_valid.min()}, '
              f'max={n_valid.max()}, mean={n_valid.mean():.1f}')

        # Plot the time series of a few thermocouples vs day_obs/seq_num
        sample_tcs = tc_cols[:5]
        fig, ax = plt.subplots(figsize=(9, 4))
        x = np.arange(len(df))
        for c in sample_tcs:
            ax.plot(x, df[c].values, '.', markersize=2, label=c.replace('m1m3_tc_', ''))
        ax.set_xlabel('Visit index (sorted by day_obs, seq_num)')
        ax.set_ylabel('Temperature [C]')
        ax.set_title(f'Sample M1M3 thermocouples — {Path(path).stem}')
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3)
        plt.show()